# 模块概述

WtUftCore 是 WonderTrader UFT（Ultra Fast Trading，极速交易）策略运行的核心模块，负责提供UFT策略的实时交易环境。主要包括：
- UFT策略引擎和策略上下文管理
- 实时市场数据处理和分发
- 交易适配和订单管理
- 行情解析器适配
- 策略参数共享管理
- 事件通知机制
- 数据结构和辅助工具类

1. **数据定义层**（UftDataDefs）：
   - 定义UFT策略使用的核心数据结构
   - 包括持仓明细、订单、成交、回合等数据结构
   - 采用块头+数据数组的结构，支持内存映射文件存储
   - 是整个模块的数据基础

2. **引擎层**（WtUftEngine + WtUftTicker）：
   - WtUftEngine：UFT引擎核心，管理策略上下文、数据订阅、时间管理等
   - WtUftTicker：实时ticker，处理实时行情并触发分钟线闭合事件
   - 是整个框架的控制中枢

3. **策略层**（UftStrategyMgr + UftStraContext）：
   - UftStrategyMgr：策略管理器，动态加载策略工厂，创建策略实例
   - UftStraContext：策略上下文，管理策略的交易上下文、持仓、订单、数据订阅等
   - 使用UftDataDefs定义的数据结构存储持仓、订单、成交等数据
   - 提供策略运行环境和交易接口

4. **数据层**（WtUftDtMgr）：
   - 管理实时tick、历史tick、K线等市场数据
   - 实现IDataManager接口，提供统一的数据查询接口
   - 支持数据订阅和数据切片查询

5. **适配器层**（TraderAdapter + ParserAdapter）：
   - TraderAdapter：交易适配器，适配不同的交易接口，提供统一的交易操作
   - ParserAdapter：行情解析器适配器，适配不同的行情数据源，统一行情接口

6. **工具支持层**（EventNotifier + ActionPolicyMgr + ShareManager + WtHelper）：
   - EventNotifier：事件通知器，通过消息队列异步广播交易事件
   - ActionPolicyMgr：动作策略管理器，管理交易动作的执行规则
   - ShareManager：共享内存管理器，管理策略参数的共享和持久化
   - WtHelper：辅助工具类，提供路径管理和时间管理功能


# 层次关系图
```mermaid
graph LR
    %% 样式定义
    classDef engineClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef contextClass fill:#fff3e0,stroke:#e65100,stroke-width:2px,color:#000;
    classDef adapterClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px,color:#000;
    classDef dataClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px,color:#000;
    classDef utilClass fill:#fce4ec,stroke:#880e4f,stroke-width:2px,color:#000;
    classDef tickerClass fill:#e0f2f1,stroke:#004d40,stroke-width:2px,color:#000;
    classDef mgrClass fill:#ffe0b2,stroke:#e65100,stroke-width:2px,color:#000;
    classDef interfaceClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;
    classDef defClass fill:#fffde7,stroke:#f57f17,stroke-width:2px,color:#000;

    %% 接口层
    subgraph Interfaces["接口层 - 抽象接口"]
        direction TB
        IUftStraCtx["IUftStraCtx<br/>UFT策略上下文接口<br/>• 交易接口<br/>• 数据查询接口<br/>• 持仓管理接口"]
        ITrdNotifySink["ITrdNotifySink<br/>交易通知接口<br/>• 成交回报<br/>• 订单回报<br/>• 持仓更新<br/>• 通道状态"]
        IParserStub["IParserStub<br/>行情解析器存根接口<br/>• Tick推送<br/>• 订单队列推送<br/>• 订单明细推送<br/>• 成交明细推送"]
        IDataManager["IDataManager<br/>数据管理器接口<br/>• Tick切片查询<br/>• K线切片查询<br/>• Level-2数据查询"]
        ITraderSpi["ITraderSpi<br/>交易接口回调<br/>• 连接回调<br/>• 登录回调<br/>• 订单回调<br/>• 成交回调"]
        IParserSpi["IParserSpi<br/>行情解析器回调<br/>• Tick回调<br/>• 订单队列回调<br/>• 订单明细回调<br/>• 成交明细回调"]
    end

    %% 数据定义层
    subgraph DataDefs["数据定义层 - 数据结构"]
        direction TB
        UftDataDefs["UftDataDefs<br/>UFT数据定义<br/>• BlockHeader 数据块头<br/>• DetailStruct 持仓明细<br/>• PositionBlock 持仓块<br/>• OrderStruct 订单结构<br/>• OrderBlock 订单块<br/>• TradeStruct 成交结构<br/>• TradeBlock 成交块<br/>• RoundStruct 回合结构<br/>• RoundBlock 回合块"]:::defClass
    end

    %% 引擎层
    subgraph Engines["引擎层 - 核心控制"]
        direction TB
        WtUftEngine["WtUftEngine<br/>UFT引擎<br/>• 策略上下文管理<br/>• 数据订阅管理<br/>• 数据分发<br/>• 时间管理<br/>• 交易日管理"]:::engineClass
        WtUftTicker["WtUftTicker<br/>实时Ticker<br/>• 实时行情处理<br/>• 分钟线闭合判断<br/>• 交易日判断<br/>• 后台定时检查"]:::tickerClass
    end

    %% 策略管理层
    subgraph StrategyMgrs["策略管理层 - 策略生命周期"]
        direction TB
        UftStrategyMgr["UftStrategyMgr<br/>策略管理器<br/>• 加载策略工厂<br/>• 创建策略实例<br/>• 管理策略映射"]:::mgrClass
        UftStraContext["UftStraContext<br/>策略上下文<br/>• 交易接口实现<br/>• 本地持仓管理<br/>• 订单管理<br/>• 数据订阅管理<br/>• 事件转发"]:::contextClass
    end

    %% 数据管理层
    subgraph DataLayer["数据管理层 - 市场数据"]
        direction TB
        WtUftDtMgr["WtUftDtMgr<br/>数据管理器<br/>• 实时Tick缓存<br/>• 历史Tick缓存<br/>• K线缓存<br/>• 数据切片查询"]:::dataClass
    end

    %% 适配器层
    subgraph Adapters["适配器层 - 外部接口适配"]
        direction TB
        TraderAdapter["TraderAdapter<br/>交易适配器<br/>• 交易接口适配<br/>• 订单管理<br/>• 持仓管理<br/>• 动作策略转换<br/>• 风险控制"]:::adapterClass
        ParserAdapter["ParserAdapter<br/>行情解析器适配器<br/>• 行情接口适配<br/>• 数据过滤<br/>• 数据标准化<br/>• 数据转发"]:::adapterClass
    end

    %% 工具支持层
    subgraph Utils["工具支持层 - 辅助功能"]
        direction TB
        EventNotifier["EventNotifier<br/>事件通知器<br/>• 消息队列集成<br/>• 异步事件处理<br/>• JSON格式转换<br/>• 事件广播"]:::utilClass
        ActionPolicyMgr["ActionPolicyMgr<br/>动作策略管理器<br/>• 交易动作规则<br/>• 品种规则映射<br/>• 手数限制管理"]:::utilClass
        ShareManager["ShareManager<br/>共享内存管理器<br/>• 参数共享域管理<br/>• 参数读写<br/>• 参数监控<br/>• 参数同步"]:::utilClass
        WtHelper["WtHelper<br/>辅助工具类<br/>• 路径管理<br/>• 时间管理<br/>• 目录创建"]:::utilClass
    end

    %% 继承关系
    WtUftEngine -.->|"实现"| IParserStub
    UftStraContext -.->|"实现"| IUftStraCtx
    UftStraContext -.->|"实现"| ITrdNotifySink
    WtUftDtMgr -.->|"实现"| IDataManager
    TraderAdapter -.->|"实现"| ITraderSpi
    ParserAdapter -.->|"实现"| IParserSpi

    %% 核心组合关系
    WtUftEngine -->|"包含"| WtUftTicker
    WtUftEngine -->|"管理"| UftStraContext
    WtUftEngine -->|"使用"| WtUftDtMgr
    WtUftEngine -->|"使用"| EventNotifier
    
    UftStrategyMgr -->|"创建"| UftStraContext
    UftStraContext -->|"使用"| TraderAdapter
    UftStraContext -->|"使用"| ShareManager
    UftStraContext -->|"使用数据结构"| UftDataDefs
    
    WtUftTicker -->|"触发事件"| WtUftEngine
    
    %% 数据流关系
    ParserAdapter -->|"推送行情"| WtUftEngine
    WtUftEngine -->|"分发数据"| UftStraContext
    WtUftEngine -->|"更新数据"| WtUftDtMgr
    
    %% 交易流关系
    UftStraContext -->|"下单"| TraderAdapter
    TraderAdapter -->|"交易回报"| UftStraContext
    TraderAdapter -->|"使用"| ActionPolicyMgr
    TraderAdapter -->|"通知事件"| EventNotifier
    
    %% 工具层关系
    WtUftEngine -.->|"使用"| WtHelper
    UftStraContext -.->|"使用"| WtHelper
    WtUftDtMgr -.->|"使用"| WtHelper
    TraderAdapter -.->|"使用"| WtHelper
    ParserAdapter -.->|"使用"| WtHelper
    
    %% 应用样式
    class WtUftEngine engineClass
    class UftStraContext contextClass
    class TraderAdapter,ParserAdapter adapterClass
    class WtUftDtMgr dataClass
    class EventNotifier,ActionPolicyMgr,ShareManager,WtHelper utilClass
    class WtUftTicker tickerClass
    class UftStrategyMgr mgrClass
    class IUftStraCtx,ITrdNotifySink,IParserStub,IDataManager,ITraderSpi,IParserSpi interfaceClass
    class UftDataDefs defClass
```


# 适配器层

## ParserAdapter.h/cpp — 行情解析器适配器

### 框架图
```mermaid
graph LR
    %% 样式定义
    classDef parserInterface fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef parserAdapter fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef parserMgr fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#000;
    classDef execInterface fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef execUnit fill:#fff9c4,stroke:#f57f17,stroke-width:3px,color:#000;
    classDef execFactory fill:#fce4ec,stroke:#c2185b,stroke-width:2px,color:#000;
    classDef external fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% =======================
    %% 外部接口层
    %% =======================
    subgraph ExternalLayer["外部接口层"]
        direction TB
        IParserApi["IParserApi<br/>行情解析器API<br/>数据源接口"]
        IParserSpi["IParserSpi<br/>行情解析器回调接口<br/>接收数据回调"]
        WtUftEngine["WtUftEngine<br/>UFT引擎<br/>数据接收者"]:::external
    end

    %% =======================
    %% 行情解析适配器层
    %% =======================
    subgraph ParserLayer["行情解析适配器层 - ParserAdapter"]
        direction TB
        IParserStub["IParserStub<br/>数据推送接口<br/>定义数据接收接口"]:::parserInterface
        ParserAdapter["ParserAdapter<br/>行情解析器适配器<br/>适配不同数据源<br/>数据过滤与转发"]:::parserAdapter
        ParserAdapterMgr["ParserAdapterMgr<br/>适配器管理器<br/>管理多个适配器"]:::parserMgr
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    ParserAdapter -.->|"实现"| IParserSpi
    WtUftEngine -.->|"实现"| IParserStub

    %% =======================
    %% 组合与管理关系（实线）
    %% =======================
    ParserAdapterMgr -->|"管理"| ParserAdapter
    ParserAdapter -->|"使用"| IParserApi
    ParserAdapter -->|"推送数据"| IParserStub
    IParserStub -->|"数据接收者"| WtUftEngine


    %% =======================
    %% 数据流关系
    %% =======================
    IParserApi -->|"数据回调"| ParserAdapter
    ParserAdapter -->|"过滤转发"| WtUftEngine

    %% 应用样式
    class IParserStub parserInterface
    class ParserAdapter parserAdapter
    class ParserAdapterMgr parserMgr
    class ExecuteContext execInterface
    class ExecuteUnit execUnit
    class IExecuterFact execFactory
    class IParserApi,IParserSpi,WtUftEngine external
```

### 解析器存根接口类 IParserStub
```cpp
class IParserStub
```

#### 推送行情数据 handle_push_quote

#### 推送委托明细数据 handle_push_order_detail

#### 推送委托队列数据 handle_push_order_queue

#### 推送逐笔成交数据 handle_push_transaction

### 解析器适配器类 ParserAdapter
```cpp
class ParserAdapter : public IParserSpi,  // 继承解析器SPI接口
	private boost::noncopyable  // 继承boost::noncopyable，禁止拷贝构造和赋值
```

#### 成员
- **核心接口指针**
  - `IParserApi* _parser_api`：行情解析器API指针，用于调用解析器功能（初始化、订阅、连接等）
  - `FuncDeleteParser _remover`：删除解析器函数指针，用于释放动态加载的解析器模块

- **状态标志**
  - `bool _stopped`：是否已停止标志，用于控制数据接收和处理流程

- **数据过滤器**
  - `ExchgFilter _exchg_filter`：交易所过滤器
    - `typedef wt_hashset<std::string> ExchgFilter`：交易所代码集合
    - 仅接收指定交易所的数据
  - `ExchgFilter _code_filter`：合约代码过滤器
    - `typedef wt_hashset<std::string> ExchgFilter`：合约代码或品种代码集合
    - 仅接收指定合约或品种的数据

- **外部依赖指针**
  - `IBaseDataMgr* _bd_mgr`：基础数据管理器指针，用于获取合约信息、查询合约列表等
  - `IParserStub* _stub`：行情数据存根接口指针，用于接收并转发解析后的行情数据（Tick、委托队列、委托明细、逐笔成交）

- **配置与标识**
  - `WTSVariant* _cfg`：配置参数指针，包含解析器模块路径、订阅配置、过滤器设置等
  - `std::string _id`：适配器ID，唯一标识符

#### 初始化与生命周期管理

##### 初始化适配器（从配置文件）init

##### 初始化适配器（外部API）initExt

##### 启动解析器 run

#### IParserSpi接口回调 - 行情数据回调

##### 处理合约列表 handleSymbolList

##### 处理实时行情（Tick数据）handleQuote

##### 处理委托队列数据（股票level2）handleOrderQueue

##### 处理逐笔委托数据（股票level2）handleOrderDetail

##### 处理逐笔成交数据 handleTransaction

#### 辅助方法

##### 处理解析器日志 handleParserLog

##### 获取基础数据管理器 getBaseDataMgr

### 解析器适配器管理器类 ParserAdapterMgr
```cpp
class ParserAdapterMgr : private boost::noncopyable  // 继承boost::noncopyable，禁止拷贝构造和赋值
```

#### 成员
`ParserAdapterMap _adapters`：解析器适配器映射表，键为解析器ID，值为适配器指针
- typedef wt_hashmap\<std::string, `ParserAdapterPtr`\> ParserAdapterMap
- typedef std::shared_ptr\<`ParserAdapter`\> ParserAdapterPtr

#### 方法

##### 添加适配器 addAdapter

##### 获取适配器 getAdapter

##### 启动所有适配器 run

## TraderAdapter.h/cpp — 交易适配器

### 框架图
```mermaid
graph LR
    %% 样式定义
    classDef traderInterface fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef traderAdapter fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef traderMgr fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#000;
    classDef notifyInterface fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef policyMgr fill:#fce4ec,stroke:#c2185b,stroke-width:2px,color:#000;
    classDef context fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#000;
    classDef external fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% =======================
    %% 外部接口层
    %% =======================
    subgraph ExternalLayer["外部接口层"]
        direction TB
        ITraderApi["ITraderApi<br/>交易接口API<br/>交易通道接口"]
        ITraderSpi["ITraderSpi<br/>交易接口回调接口<br/>接收交易回调"]
        UftStraContext["UftStraContext<br/>策略上下文<br/>交易请求者"]:::context
    end

    %% =======================
    %% 交易适配器层
    %% =======================
    subgraph TraderLayer["交易适配器层 - TraderAdapter"]
        direction TB
        ITrdNotifySink["ITrdNotifySink<br/>交易通知接口<br/>定义交易事件接收接口"]:::notifyInterface
        TraderAdapter["TraderAdapter<br/>交易适配器<br/>适配不同交易接口<br/>订单与持仓管理<br/>风险控制"]:::traderAdapter
        TraderAdapterMgr["TraderAdapterMgr<br/>适配器管理器<br/>管理多个适配器"]:::traderMgr
    end

    %% =======================
    %% 工具支持层
    %% =======================
    subgraph UtilsLayer["工具支持层"]
        direction TB
        ActionPolicyMgr["ActionPolicyMgr<br/>动作策略管理器<br/>交易动作规则管理"]:::policyMgr
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    TraderAdapter -.->|"实现"| ITraderSpi
    UftStraContext -.->|"实现"| ITrdNotifySink

    %% =======================
    %% 组合与管理关系（实线）
    %% =======================
    TraderAdapterMgr -->|"管理"| TraderAdapter
    TraderAdapter -->|"使用"| ITraderApi
    TraderAdapter -->|"使用"| ActionPolicyMgr
    TraderAdapter -->|"通知"| ITrdNotifySink
    ITrdNotifySink -->|"数据接收者"| UftStraContext

    %% =======================
    %% 交易流关系
    %% =======================
    UftStraContext -->|"下单请求"| TraderAdapter
    ITraderApi -->|"交易回调"| TraderAdapter
    TraderAdapter -->|"交易回报"| UftStraContext

    %% 应用样式
    class ITrdNotifySink notifyInterface
    class TraderAdapter traderAdapter
    class TraderAdapterMgr traderMgr
    class ActionPolicyMgr policyMgr
    class UftStraContext context
    class ITraderApi,ITraderSpi external
```

### 交易适配器类 TraderAdapter
```cpp
class TraderAdapter : public ITraderSpi
```

#### 成员
- **配置与标识**
  - `WTSVariant* _cfg`：配置参数
  - `std::string _id`：适配器ID
  - `std::string _order_pattern`：订单用户标签模式
  - `uint32_t _trading_day`：交易日
- **核心接口指针**
  - `ITraderApi* _trader_api`：交易接口指针
  - `FuncDeleteTrader _remover`：删除交易接口的函数指针
- **状态管理**
  - `AdapterState _state`：适配器状态（枚举类型）
    ```cpp
    /* 定义交易通道从连接、登录到就绪的各个状态*/
    typedef enum tagAdapterState
    {
      AS_NOTLOGIN, // 未登录状态：初始状态，尚未开始登录流程
      AS_LOGINING, // 正在登录：已发起登录请求，等待登录结果
      AS_LOGINED, // 已登录：登录成功，但尚未完成数据查询
      AS_LOGINFAILED, // 登录失败：登录请求被拒绝或失败
      AS_POSITION_QRYED, // 仓位已查：持仓查询完成
      AS_ORDERS_QRYED, // 订单已查：订单查询完成
      AS_TRADES_QRYED, // 成交已查：成交查询完成
      AS_ALLREADY // 全部就绪：所有查询完成，交易通道可以使用
    }
    ```
- **外部依赖指针**
  - `wt_hashset<ITrdNotifySink*> _sinks`：通知接收器集合
  - `IBaseDataMgr* _bd_mgr`：基础数据管理器指针
  - `ActionPolicyMgr* _policy_mgr`：动作策略管理器指针
- **持仓管理**
  - `wt_hashmap<std::string, PosItem> _positions`：持仓映射表，键为合约代码
    ```cpp
    /* @brief 持仓项结构体
      * 用于存储单个合约的持仓信息，包括多空两个方向的今昨持仓数据*/
    typedef struct _PosItem
    {
      // 多仓数据（做多方向持仓）
      double l_newvol; // 多头今仓数量：今日开仓的多头持仓数量
      double l_newavail; // 多头今仓可用：今日开仓的多头持仓中可用于平仓的数量
      double l_prevol; // 多头昨仓数量：昨日及之前开仓的多头持仓数量
      double l_preavail; // 多头昨仓可用：昨日及之前开仓的多头持仓中可用于平仓的数量
      // 空仓数据（做空方向持仓）
      double s_newvol; // 空头今仓数量：今日开仓的空头持仓数量
      double s_newavail; // 空头今仓可用：今日开仓的空头持仓中可用于平仓的数量
      double s_prevol; // 空头昨仓数量：昨日及之前开仓的空头持仓数量
      double s_preavail; // 空头昨仓可用：昨日及之前开仓的空头持仓中可用于平仓的数量
    } PosItem;
    ```
- **订单管理**
  - `SpinMutex _mtx_orders`：订单列表互斥锁
  - `OrderMap* _orders`：订单映射表
    - `typedef WTSMap<uint32_t> OrderMap`：订单映射表类型
  - `wt_hashset<std::string> _orderids`：订单号集合，主要用于标记是否处理过该订单
  - `wt_hashmap<std::string, double> _undone_qty`：未完成数量映射表，键为合约代码
- **交易统计**
  - `TradeStatMap* _stat_map`：交易统计映射表，键为合约代码
    - `typedef WTSHashMap<std::string> TradeStatMap`：交易统计映射表类型
- **风险控制时间缓存**
  - `CodeTimeCacheMap _order_time_cache`：下单时间缓存，键为合约代码，值为时间戳列表
  - `CodeTimeCacheMap _cancel_time_cache`：撤单时间缓存，键为合约代码，值为时间戳列表
    - `typedef std::vector<uint64_t> TimeCacheList`：时间缓存列表类型
    - `typedef wt_hashmap<std::string, TimeCacheList> CodeTimeCacheMap`：代码时间缓存映射表类型
- **风险控制**
  - `wt_hashset<std::string> _exclude_codes`：被风控排除的合约代码集合
  - `RiskParamsMap _risk_params_map`：风险参数映射表，键为品种代码
    - `typedef wt_hashmap<std::string, RiskParams> RiskParamsMap`：风险参数映射表类型
      ```cpp
      /* @brief 风控参数结构体
      * 定义交易风控策略的参数，包括下单和撤单的频率限制 */
      typedef struct _RiskParams
      {
        uint32_t _order_times_boundary; // 下单频率边界：在统计时间窗口内允许的最大下单次数
        uint32_t _order_stat_timespan; // 下单统计时间窗口：统计下单频率的时间跨度（秒）
        uint32_t _order_total_limits; // 下单总限额：当日允许的最大下单总次数

        uint32_t _cancel_times_boundary; // 撤单频率边界：在统计时间窗口内允许的最大撤单次数
        uint32_t _cancel_stat_timespan; // 撤单统计时间窗口：统计撤单频率的时间跨度（秒）
        uint32_t _cancel_total_limits; // 撤单总限额：当日允许的最大撤单总次数
      } RiskParams;
      ```
  - `bool _risk_mon_enabled`：是否启用风险监控

#### 初始化与生命周期管理

#### 初始化与生命周期管理

##### 初始化交易适配器（从配置文件）init

##### 初始化交易适配器（外部API）initExt

##### 启动交易适配器 run

#### 持仓和订单管理

##### 获取持仓数量 getPosition

##### 枚举持仓并通知接收器 enumPosition

##### 获取订单列表 getOrders

##### 获取未完成数量 getUndoneQty

##### 获取交易统计信息数量 getInfos

#### 交易操作

##### 买入操作 buy

##### 卖出操作 sell

##### 开多单 openLong

##### 开空单 openShort

##### 平多单 closeLong

##### 平空单 closeShort

##### 撤单 cancel

##### 全部撤单 cancelAll

#### 风险控制

##### 检查合约是否允许交易 isTradeEnabled

##### 检查撤单限制 checkCancelLimits

##### 检查下单限制 checkOrderLimits

#### ITraderSpi接口

##### 处理交易事件 handleEvent

##### 登录结果回调 onLoginResult

##### 登出回调 onLogout

##### 委托回报回调 onRspEntrust

##### 账户查询回调 onRspAccount

##### 持仓查询回调 onRspPosition

##### 订单查询回调 onRspOrders

##### 成交查询回调 onRspTrades

##### 订单推送回调 onPushOrder

##### 成交推送回调 onPushTrade

##### 交易错误回调 onTraderError

##### 获取基础数据管理器 getBaseDataMgr

##### 处理交易日志 handleTraderLog

#### 内部方法

##### 执行委托下单 doEntrust

##### 执行撤单 doCancel

##### 获取合约信息 getContract

##### 更新未完成数量 updateUndone

##### 获取风险控制参数 getRiskParams

### 交易适配器管理器类 TraderAdapterMgr
```cpp
class TraderAdapterMgr : private boost::noncopyable
```

#### 成员
- `TraderAdapterMap _adapters`：交易适配器映射表，键为交易通道名称
  - `typedef wt_hashmap<std::string, TraderAdapterPtr> TraderAdapterMap`：交易适配器映射表类型
  - `typedef std::shared_ptr<TraderAdapter> TraderAdapterPtr`：交易适配器智能指针类型

#### 添加适配器 addAdapter

#### 获取指定名称的适配器 getAdapter

#### 获取适配器映射表 getAdapters

#### 启动所有适配器 run

# 策略管理层

## 框架图
```mermaid
graph LR
    %% 样式定义
    classDef dataStruct fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef dataBlock fill:#bbdefb,stroke:#0d47a1,stroke-width:2px,color:#000;
    classDef context fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef wrapper fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#000;
    classDef manager fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef interface fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;
    classDef strategy fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#000;

    %% =======================
    %% 数据定义层 - UftDataDefs.h
    %% =======================
    subgraph DataLayer["数据定义层 - UftDataDefs.h"]
        direction TB
        
        subgraph BaseStruct["基础结构"]
            BlockHeader["BlockHeader<br/>数据块头<br/>标志/类型/日期/容量/大小"]:::dataStruct
        end
        
        subgraph DataStructs["数据结构"]
            DetailStruct["DetailStruct<br/>持仓明细结构<br/>交易所/合约/方向/数量/价格/盈亏"]:::dataStruct
            OrderStruct["OrderStruct<br/>订单结构<br/>交易所/合约/方向/开平/数量/价格/状态"]:::dataStruct
            TradeStruct["TradeStruct<br/>成交结构<br/>交易所/合约/方向/开平/数量/价格/时间"]:::dataStruct
            RoundStruct["RoundStruct<br/>回合结构<br/>交易所/合约/方向/开平价格/盈亏"]:::dataStruct
        end
        
        subgraph DataBlocks["数据块"]
            PositionBlock["PositionBlock<br/>持仓数据块<br/>继承BlockHeader<br/>包含DetailStruct数组"]:::dataBlock
            OrderBlock["OrderBlock<br/>订单数据块<br/>继承BlockHeader<br/>包含OrderStruct数组"]:::dataBlock
            TradeBlock["TradeBlock<br/>成交数据块<br/>继承BlockHeader<br/>包含TradeStruct数组"]:::dataBlock
            RoundBlock["RoundBlock<br/>回合数据块<br/>继承BlockHeader<br/>包含RoundStruct数组"]:::dataBlock
        end
    end

    %% =======================
    %% 策略上下文层 - UftStraContext.h/cpp
    %% =======================
    subgraph ContextLayer["策略上下文层 - UftStraContext"]
        direction TB
        
        subgraph ContextClass["上下文类"]
            UftStraContext["UftStraContext<br/>策略上下文<br/>实现IUftStraCtx和ITrdNotifySink<br/>管理持仓/订单/成交/回合<br/>数据持久化与事件转发"]:::context
        end
        
        subgraph BlockPairs["数据块配对"]
            PosBlkPair["PosBlkPair<br/>持仓数据块配对<br/>PositionBlock + 内存映射文件 + 锁"]:::dataBlock
            OrdBlkPair["OrdBlkPair<br/>订单数据块配对<br/>OrderBlock + 内存映射文件 + 锁"]:::dataBlock
            TrdBlkPair["TrdBlkPair<br/>成交数据块配对<br/>TradeBlock + 内存映射文件 + 锁"]:::dataBlock
            RndBlkPair["RndBlkPair<br/>回合数据块配对<br/>RoundBlock + 内存映射文件 + 锁"]:::dataBlock
        end
    end

    %% =======================
    %% 策略管理层 - UftStrategyMgr.h/cpp
    %% =======================
    subgraph ManagerLayer["策略管理层 - UftStrategyMgr"]
        direction TB
        
        subgraph WrapperClass["包装器类"]
            UftStraWrapper["UftStraWrapper<br/>策略包装器<br/>包装策略实例和工厂<br/>管理策略生命周期"]:::wrapper
        end
        
        subgraph ManagerClass["管理器类"]
            UftStrategyMgr["UftStrategyMgr<br/>策略管理器<br/>加载策略工厂<br/>创建和管理策略实例"]:::manager
        end
    end

    %% =======================
    %% 外部接口层
    %% =======================
    subgraph InterfaceLayer["外部接口层"]
        direction TB
        IUftStraCtx["IUftStraCtx<br/>策略上下文接口<br/>交易接口/数据查询/持仓管理"]:::interface
        ITrdNotifySink["ITrdNotifySink<br/>交易通知接口<br/>成交回报/订单回报/持仓更新"]:::interface
        UftStrategy["UftStrategy<br/>策略基类<br/>定义策略生命周期回调"]:::strategy
        IUftStrategyFact["IUftStrategyFact<br/>策略工厂接口<br/>创建/删除策略实例"]:::interface
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    PositionBlock -.->|"继承"| BlockHeader
    OrderBlock -.->|"继承"| BlockHeader
    TradeBlock -.->|"继承"| BlockHeader
    RoundBlock -.->|"继承"| BlockHeader
    
    UftStraContext -.->|"实现"| IUftStraCtx
    UftStraContext -.->|"实现"| ITrdNotifySink
    UftStrategy -.->|"由工厂创建"| IUftStrategyFact

    %% =======================
    %% 组合关系（实线）
    %% =======================
    PositionBlock -->|"包含"| DetailStruct
    OrderBlock -->|"包含"| OrderStruct
    TradeBlock -->|"包含"| TradeStruct
    RoundBlock -->|"包含"| RoundStruct
    
    PosBlkPair -->|"使用"| PositionBlock
    OrdBlkPair -->|"使用"| OrderBlock
    TrdBlkPair -->|"使用"| TradeBlock
    RndBlkPair -->|"使用"| RoundBlock
    
    UftStraContext -->|"管理"| PosBlkPair
    UftStraContext -->|"管理"| OrdBlkPair
    UftStraContext -->|"管理"| TrdBlkPair
    UftStraContext -->|"管理"| RndBlkPair
    UftStraContext -->|"绑定"| UftStrategy
    
    UftStraWrapper -->|"包装"| UftStrategy
    UftStraWrapper -->|"持有"| IUftStrategyFact
    UftStrategyMgr -->|"管理"| UftStraWrapper
    UftStrategyMgr -->|"加载"| IUftStrategyFact

    %% =======================
    %% 数据流关系
    %% =======================
    UftStrategy -->|"使用"| UftStraContext
    UftStraContext -->|"事件转发"| UftStrategy
    UftStrategyMgr -->|"创建策略"| UftStraWrapper

    %% 应用样式
    class BlockHeader,DetailStruct,OrderStruct,TradeStruct,RoundStruct dataStruct
    class PositionBlock,OrderBlock,TradeBlock,RoundBlock,PosBlkPair,OrdBlkPair,TrdBlkPair,RndBlkPair dataBlock
    class UftStraContext context
    class UftStraWrapper wrapper
    class UftStrategyMgr manager
    class IUftStraCtx,ITrdNotifySink,IUftStrategyFact interface
    class UftStrategy strategy
```

## UftDataDefs.h — 数据定义

## UftStraContext.h/cpp — 策略上下文

### UFT策略上下文类 UftStraContext
```cpp
class UftStraContext : public IUftStraCtx, public ITrdNotifySink
```

## UftStrategyMgr.h/cpp — 策略管理器

### UFT策略包装器类 UftStraWrapper
```cpp
class UftStraWrapper
```

### UFT策略管理器类 UftStrategyMgr
```cpp
class UftStrategyMgr : private boost::noncopyable
```

# 数据管理器 WtUftDtMgr.h/cpp

## 框架图

```mermaid
graph LR
    %% 样式定义
    classDef interface fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef manager fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef cache fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef notify fill:#fce4ec,stroke:#c2185b,stroke-width:2px,color:#000;
    classDef external fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% =======================
    %% 接口层
    %% =======================
    subgraph InterfaceLayer["接口层"]
        direction TB
        IDataManager["IDataManager<br/>数据管理器接口<br/>定义数据查询接口"]:::interface
    end

    %% =======================
    %% 数据管理器层 - WtUftDtMgr.h/cpp
    %% =======================
    subgraph ManagerLayer["数据管理器层 - WtUftDtMgr"]
        direction TB
        
        subgraph ManagerClass["管理器类"]
            WtUftDtMgr["WtUftDtMgr<br/>UFT数据管理器<br/>实现IDataManager接口<br/>• 实时行情处理<br/>• 数据缓存管理<br/>• 数据查询接口<br/>• 数据订阅管理"]:::manager
        end
        
        subgraph CacheLayer["数据缓存层"]
            RtTickMap["_rt_tick_map<br/>实时Tick缓存<br/>存储最新Tick数据"]:::cache
            TicksCache["_ticks_cache<br/>历史Tick缓存<br/>存储历史Tick数据"]:::cache
            BarsCache["_bars_cache<br/>K线缓存<br/>存储K线数据"]:::cache
        end
        
        subgraph Subscription["订阅管理"]
            SubedBasicBars["_subed_basic_bars<br/>已订阅基础K线集合"]:::cache
        end
        
        subgraph NotifyStruct["通知结构"]
            NotifyItem["NotifyItem<br/>K线通知项"]:::notify
            BarNotifies["_bar_notifies<br/>K线通知项列表"]:::notify
        end
    end

    %% =======================
    %% 外部依赖层
    %% =======================
    subgraph ExternalLayer["外部依赖层"]
        direction TB
        WtUftEngine["WtUftEngine<br/>UFT引擎<br/>数据管理器使用者"]:::external
        WTSDataFactory["WTSDataFactory<br/>数据工厂<br/>创建数据对象"]:::external
        WTSVariant["WTSVariant<br/>配置变体类<br/>配置参数"]:::external
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    WtUftDtMgr -.->|"实现"| IDataManager

    %% =======================
    %% 组合关系（实线）
    %% =======================
    WtUftDtMgr -->|"管理"| RtTickMap
    WtUftDtMgr -->|"管理"| TicksCache
    WtUftDtMgr -->|"管理"| BarsCache
    WtUftDtMgr -->|"管理"| SubedBasicBars
    WtUftDtMgr -->|"管理"| BarNotifies
    BarNotifies -->|"包含"| NotifyItem

    %% =======================
    %% 数据流关系
    %% =======================
    WtUftEngine -->|"使用"| WtUftDtMgr
    WtUftEngine -->|"推送行情"| WtUftDtMgr
    WTSDataFactory -->|"创建数据"| WtUftDtMgr
    WTSVariant -->|"配置"| WtUftDtMgr

    %% 应用样式
    class IDataManager interface
    class WtUftDtMgr manager
    class RtTickMap,TicksCache,BarsCache,SubedBasicBars cache
    class WtUftEngine,WTSDataFactory,WTSVariant external
    class NotifyItem,BarNotifies notify
```

## UFT数据管理器类 WtUftDtMgr
```cpp
class WtUftDtMgr : public IDataManager
```

### 成员

- **核心引擎指针**
  - `WtUftEngine* _engine`：UFT引擎指针

- **数据订阅管理**
  - `wt_hashset<std::string> _subed_basic_bars`：已订阅的基础K线集合

- **数据缓存映射表**
  - `DataCacheMap* _bars_cache`：K线缓存映射表
    - `typedef WTSHashMap<std::string> DataCacheMap`：数据缓存映射表类型
  - `DataCacheMap* _ticks_cache`：历史Tick缓存映射表
    - `typedef WTSHashMap<std::string> DataCacheMap`：数据缓存映射表类型
  - `DataCacheMap* _rt_tick_map`：实时tick缓存映射表
    - `typedef WTSHashMap<std::string> DataCacheMap`：数据缓存映射表类型

- **K线通知管理**
  - `std::vector<NotifyItem> _bar_notifies`：K线通知项列表
    ```cpp
    /* @brief K线通知项结构体
    * 用于存储K线更新通知信息。*/
    typedef struct _NotifyItem
    {
        std::string _code; // 合约代码
        std::string _period; // 周期字符串
        uint32_t _times; // 周期倍数
        WTSBarStruct* _newBar; // 新的K线数据指针
    } NotifyItem;
    ```

### 初始化 init

### 处理行情推送 handle_push_quote

### IDataManager接口

#### 获取Tick数据切片 get_tick_slice

#### 获取订单队列数据切片 get_order_queue_slice

#### 获取订单明细数据切片 get_order_detail_slice

#### 获取成交明细数据切片 get_transaction_slice

#### 获取K线数据切片 get_kline_slice

#### 获取最新Tick数据 grab_last_tick

# 引擎层

## 框架图

```mermaid
graph LR
    %% 样式定义
    classDef interface fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef engine fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef ticker fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef subscription fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#000;
    classDef context fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#000;
    classDef time fill:#e0f2f1,stroke:#004d40,stroke-width:2px,color:#000;
    classDef external fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% =======================
    %% 接口层
    %% =======================
    subgraph InterfaceLayer["接口层"]
        direction TB
        IParserStub["IParserStub<br/>行情解析器存根接口<br/>定义数据推送接口"]:::interface
    end

    %% =======================
    %% 引擎层 - WtUftEngine.h/cpp
    %% =======================
    subgraph EngineLayer["引擎层 - WtUftEngine"]
        direction TB
        
        subgraph EngineClass["引擎类"]
            WtUftEngine["WtUftEngine<br/>UFT引擎<br/>实现IParserStub接口<br/>• 策略上下文管理<br/>• 数据订阅管理<br/>• 数据分发<br/>• 时间管理<br/>• 交易日管理"]:::engine
        end
        
        subgraph SubscriptionLayer["订阅管理层"]
            TickSubMap["_tick_sub_map<br/>Tick订阅映射表<br/>合约代码 → 策略上下文ID集合"]:::subscription
            OrdQueSubMap["_ordque_sub_map<br/>订单队列订阅映射表<br/>合约代码 → 策略上下文ID集合"]:::subscription
            OrdDtlSubMap["_orddtl_sub_map<br/>订单明细订阅映射表<br/>合约代码 → 策略上下文ID集合"]:::subscription
            TransSubMap["_trans_sub_map<br/>成交明细订阅映射表<br/>合约代码 → 策略上下文ID集合"]:::subscription
            BarSubMap["_bar_sub_map<br/>K线订阅映射表<br/>合约代码-周期-倍数 → 策略上下文ID集合"]:::subscription
        end
        
        subgraph ContextLayer["上下文管理层"]
            ContextMap["_ctx_map<br/>策略上下文映射表<br/>策略上下文ID → 策略上下文指针"]:::context
        end
        
        subgraph TimeLayer["时间管理层"]
            CurDate["_cur_date<br/>当前日期<br/>YYYYMMDD格式"]:::time
            CurTime["_cur_time<br/>当前时间<br/>HHMMSS格式"]:::time
            CurRawTime["_cur_raw_time<br/>原始时间<br/>HHMMSS格式"]:::time
            CurSecs["_cur_secs<br/>当前秒数<br/>包含毫秒"]:::time
            CurTDate["_cur_tdate<br/>当前交易日<br/>YYYYMMDD格式"]:::time
        end
    end

    %% =======================
    %% Ticker层 - WtUftTicker.h/cpp
    %% =======================
    subgraph TickerLayer["Ticker层 - WtUftTicker"]
        direction TB
        
        subgraph TickerClass["Ticker类"]
            WtUftRtTicker["WtUftRtTicker<br/>实时Ticker<br/>• 实时行情处理<br/>• 分钟线闭合判断<br/>• 交易日判断<br/>• 后台定时检查"]:::ticker
        end
        
        subgraph TickerTimeMgr["Ticker时间管理"]
            TickerDate["_date<br/>当前日期"]:::time
            TickerTimeVal["_time<br/>当前时间"]:::time
            CurPos["_cur_pos<br/>当前分钟位置<br/>交易时段内的分钟数"]:::time
            NextCheckTime["_next_check_time<br/>下次检查时间<br/>时间戳（毫秒）"]:::time
            LastEmitPos["_last_emit_pos<br/>上次触发分钟位置"]:::time
        end
        
        subgraph TickerThreadMgr["线程管理"]
            Stopped["_stopped<br/>停止标志"]:::time
            Thread["_thrd<br/>后台线程指针"]:::time
            Mutex["_mtx<br/>互斥锁<br/>保护共享数据"]:::time
        end
    end

    %% =======================
    %% 外部依赖层
    %% =======================
    subgraph ExternalLayer["外部依赖层"]
        direction TB
        WtUftDtMgr["WtUftDtMgr<br/>数据管理器<br/>市场数据管理"]:::external
        TraderAdapterMgr["TraderAdapterMgr<br/>交易适配器管理器<br/>交易接口管理"]:::external
        EventNotifier["EventNotifier<br/>事件通知器<br/>事件广播"]:::external
        IBaseDataMgr["IBaseDataMgr<br/>基础数据管理器<br/>合约/商品信息查询"]:::external
        WTSSessionInfo["WTSSessionInfo<br/>交易时段信息<br/>交易时间模板"]:::external
        UftStraContext["UftStraContext<br/>策略上下文<br/>策略运行环境"]:::external
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    WtUftEngine -.->|"实现"| IParserStub

    %% =======================
    %% 组合关系（实线）
    %% =======================
    WtUftEngine -->|"包含"| WtUftRtTicker
    WtUftEngine -->|"管理"| TickSubMap
    WtUftEngine -->|"管理"| OrdQueSubMap
    WtUftEngine -->|"管理"| OrdDtlSubMap
    WtUftEngine -->|"管理"| TransSubMap
    WtUftEngine -->|"管理"| BarSubMap
    WtUftEngine -->|"管理"| ContextMap
    WtUftEngine -->|"管理"| CurDate
    WtUftEngine -->|"管理"| CurTime
    WtUftEngine -->|"管理"| CurRawTime
    WtUftEngine -->|"管理"| CurSecs
    WtUftEngine -->|"管理"| CurTDate
    
    WtUftRtTicker -->|"持有"| WtUftEngine
    WtUftRtTicker -->|"管理"| TickerDate
    WtUftRtTicker -->|"管理"| TickerTimeVal
    WtUftRtTicker -->|"管理"| CurPos
    WtUftRtTicker -->|"管理"| NextCheckTime
    WtUftRtTicker -->|"管理"| LastEmitPos
    WtUftRtTicker -->|"管理"| Stopped
    WtUftRtTicker -->|"管理"| Thread
    WtUftRtTicker -->|"管理"| Mutex
    WtUftRtTicker -->|"使用"| WTSSessionInfo

    %% =======================
    %% 数据流关系
    %% =======================
    IParserStub -->|"数据推送"| WtUftEngine
    WtUftEngine -->|"转发Tick"| WtUftRtTicker
    WtUftRtTicker -->|"触发分钟闭合"| WtUftEngine
    WtUftRtTicker -->|"触发交易日事件"| WtUftEngine
    WtUftEngine -->|"分发数据"| TickSubMap
    WtUftEngine -->|"分发数据"| OrdQueSubMap
    WtUftEngine -->|"分发数据"| OrdDtlSubMap
    WtUftEngine -->|"分发数据"| TransSubMap
    WtUftEngine -->|"分发数据"| BarSubMap
    TickSubMap -->|"订阅关系"| ContextMap
    OrdQueSubMap -->|"订阅关系"| ContextMap
    OrdDtlSubMap -->|"订阅关系"| ContextMap
    TransSubMap -->|"订阅关系"| ContextMap
    BarSubMap -->|"订阅关系"| ContextMap
    ContextMap -->|"管理"| UftStraContext
    
    WtUftEngine -->|"使用"| WtUftDtMgr
    WtUftEngine -->|"使用"| TraderAdapterMgr
    WtUftEngine -->|"使用"| EventNotifier
    WtUftEngine -->|"使用"| IBaseDataMgr
    WtUftRtTicker -->|"查询"| IBaseDataMgr

    %% 应用样式
    class IParserStub interface
    class WtUftEngine engine
    class WtUftRtTicker ticker
    class TickSubMap,OrdQueSubMap,OrdDtlSubMap,TransSubMap,BarSubMap subscription
    class ContextMap context
    class CurDate,CurTime,CurRawTime,CurSecs,CurTDate,TickerDate,TickerTimeVal,CurPos,NextCheckTime,LastEmitPos,Stopped,Thread,Mutex time
    class WtUftDtMgr,TraderAdapterMgr,EventNotifier,IBaseDataMgr,WTSSessionInfo,UftStraContext external
```

## WtUftTicker.h/cpp — UFT实时ticker
```cpp
class WtUftRtTicker
```

### 成员
- **核心依赖指针**
  - `WTSSessionInfo* _s_info`：交易时段信息指针
  - `WtUftEngine* _engine`：UFT引擎指针
- **时间管理**
  - `uint32_t _date`：当前日期（YYYYMMDD格式）
  - `uint32_t _time`：当前时间（HHMMSS格式）
  - `uint32_t _cur_pos`：当前分钟位置（交易时段内的分钟数）
- **线程同步与状态**
  - `StdUniqueMutex _mtx`：互斥锁，用于保护共享数据
  - `std::atomic<uint64_t> _next_check_time`：下次检查时间（时间戳，毫秒）
  - `std::atomic<uint32_t> _last_emit_pos`：上次触发的分钟位置
  - `bool _stopped`：停止标志
  - `StdThreadPtr _thrd`：后台线程指针

### 初始化与生命周期管理

#### 初始化ticker init

#### 启动ticker run

#### 停止ticker stop

### 处理Tick数据 on_tick

## WtUftEngine.h/cpp — UFT引擎
```cpp
class WtUftEngine : public IParserStub
```

### 成员
- **时间管理**
  - `uint32_t _cur_date`：当前日期（YYYYMMDD格式）
  - `uint32_t _cur_time`：当前时间（HHMMSS格式），是1分钟线时间，比如0900，这个时候的1分钟线是0901，_cur_time也就是0901，这个是为了CTA里面方便
  - `uint32_t _cur_raw_time`：当前真实时间（HHMMSS格式）
  - `uint32_t _cur_secs`：当前秒数（包含毫秒）
  - `uint32_t _cur_tdate`：当前交易日（YYYYMMDD格式）
- **核心管理器指针**
  - `IBaseDataMgr* _base_data_mgr`：基础数据管理器指针
  - `WtUftDtMgr* _data_mgr`：数据管理器指针
  - `TraderAdapterMgr* _adapter_mgr`：交易适配器管理器指针
  - `EventNotifier* _notifier`：事件通知器指针
- **数据订阅映射表**
  - `StraSubMap _tick_sub_map`：tick数据订阅表
    - `typedef wt_hashmap<std::string, SubList> StraSubMap`：策略订阅映射表类型
    - `typedef wt_hashset<uint32_t> SubList`：订阅列表类型，策略上下文ID集合
    - 键为合约代码，值为订阅该合约的策略上下文ID集合
  - `StraSubMap _ordque_sub_map`：委托队列订阅表
    - `typedef wt_hashmap<std::string, SubList> StraSubMap`：策略订阅映射表类型
    - `typedef wt_hashset<uint32_t> SubList`：订阅列表类型，策略上下文ID集合
    - 键为合约代码，值为订阅该合约的策略上下文ID集合
  - `StraSubMap _orddtl_sub_map`：委托明细订阅表
    - `typedef wt_hashmap<std::string, SubList> StraSubMap`：策略订阅映射表类型
    - `typedef wt_hashset<uint32_t> SubList`：订阅列表类型，策略上下文ID集合
    - 键为合约代码，值为订阅该合约的策略上下文ID集合
  - `StraSubMap _trans_sub_map`：成交明细订阅表
    - `typedef wt_hashmap<std::string, SubList> StraSubMap`：策略订阅映射表类型
    - `typedef wt_hashset<uint32_t> SubList`：订阅列表类型，策略上下文ID集合
    - 键为合约代码，值为订阅该合约的策略上下文ID集合
  - `StraSubMap _bar_sub_map`：K线数据订阅表（key格式：合约代码-周期-倍数）
    - `typedef wt_hashmap<std::string, SubList> StraSubMap`：策略订阅映射表类型
    - `typedef wt_hashset<uint32_t> SubList`：订阅列表类型，策略上下文ID集合
    - 键为合约代码-周期-倍数，值为订阅该K线的策略上下文ID集合
- **策略上下文管理**
  - `ContextMap _ctx_map`：策略上下文映射表
    - `typedef wt_hashmap<uint32_t, UftContextPtr> ContextMap`：策略上下文映射表类型
    - `typedef std::shared_ptr<IUftStraCtx> UftContextPtr`：UFT策略上下文智能指针类型
    - 键为策略上下文ID，值为策略上下文智能指针
- **实时Ticker与配置**
  - `WtUftRtTicker* _tm_ticker`：实时ticker指针
  - `WTSVariant* _cfg`：配置对象指针
  - `bool _dependent`：子策略独立记账标志

### 时间管理

#### 设置日期时间 set_date_time

#### 设置交易日 set_trading_date

#### 获取当前日期 get_date

#### 获取当前分钟时间 get_min_time

#### 获取原始时间 get_raw_time

#### 获取当前秒数 get_secs

#### 获取交易日 get_trading_date

### 基础数据查询

#### 获取基础数据管理器 get_basedata_mgr

#### 获取交易时段信息 get_session_info

#### 获取商品信息 get_commodity_info

#### 获取合约信息 get_contract_info

### 数据查询接口

#### 获取最新Tick数据 get_last_tick

#### 获取Tick数据切片 get_tick_slice

#### 获取K线数据切片 get_kline_slice

#### 获取订单队列数据切片 get_order_queue_slice

#### 获取订单明细数据切片 get_order_detail_slice

#### 获取成交明细数据切片 get_transaction_slice

### 数据订阅

#### 订阅Tick数据 sub_tick

#### 订阅订单队列数据 sub_order_queue

#### 订阅订单明细数据 sub_order_detail

#### 订阅成交明细数据 sub_transaction

### 初始化与生命周期管理

#### 设置交易适配器管理器 set_adapter_mgr

#### 初始化引擎 init

#### 运行引擎 run

### 引擎生命周期与事件回调

#### 初始化完成回调 on_init

#### 交易日开始回调 on_session_begin

#### 交易日结束回调 on_session_end

#### 分钟结束回调 on_minute_end

### 数据分发

#### 处理Tick数据 on_tick

#### 处理K线数据 on_bar

### IParserStub接口实现

#### 处理行情推送 handle_push_quote

#### 处理订单明细推送 handle_push_order_detail

#### 处理订单队列推送 handle_push_order_queue

#### 处理成交明细推送 handle_push_transaction

### 策略上下文管理

#### 添加策略上下文 addContext

#### 获取策略上下文 getContext

### 通知参数更新 notify_params_update

#### 获取当前价格 get_cur_price

#### 通知参数更新 notify_params_update

# 接口层

## ITrdNotifySink.h/cpp — 交易通知接口
```cpp
class ITrdNotifySink
```

### 成交回报回调 on_trade

### 订单回报回调 on_order

### 持仓更新回调 on_position

### 交易通道就绪回调 on_channel_ready

### 交易通道丢失回调 on_channel_lost

### 下单回报回调 on_entrust

# 工具支持层

## ActionPolicyMgr.h/cpp — 动作策略管理器类
```cpp
class ActionPolicyMgr
```

### 成员
- **规则映射表**
  - `RulesMap _rules`：规则表，存储所有规则组及其规则列表
    - `typedef wt_hashmap<std::string, ActionRuleGroup> RulesMap`：规则映射表类型
    - 键为规则组名称（`std::string`），值为动作规则组（`ActionRuleGroup`）
    - `typedef std::vector<ActionRule> ActionRuleGroup`：动作规则组类型，动作规则向量
      ```cpp
      /**
       * @struct ActionRule
      * @brief 动作规则结构体
      * 定义单个动作规则的详细信息，包括动作类型、手数限制等。
      */
      typedef struct _ActionRule
      {
        ActionType _atype; // 动作类型（开仓、平仓、平今、平昨）
        uint32_t _limit; // 总手数限制（多头+空头）
        uint32_t _limit_l; // 多头手数限制
        uint32_t _limit_s; // 空头手数限制
        bool _pure; // 是否纯仓标志，主要针对AT_CloseToday和AT_CloseYestoday，用于判断是否是净今仓或者净昨仓（true表示净仓，false表示允许双向持仓）
      } ActionRule;
      ```

- **品种规则映射表**
  - `wt_hashmap<std::string, std::string> _comm_rule_map`：品种规则映射表
    - 键为合约品种代码（`std::string`），值为规则组名称（`std::string`）
    - 用于将合约品种映射到对应的规则组

### 初始化动作策略管理器 init

### 获取动作规则组 getActionRules

## EventNotifier.h/cpp — 事件通知器类
```cpp
class EventNotifier
```

### 成员
- **消息队列配置**
  - `std::string _url`：消息队列URL地址
  - `uint32_t _mq_sid`：消息队列服务器ID

- **消息队列函数指针**
  - `FuncCreateMQServer _creator`：创建消息队列服务器函数指针
    - `typedef unsigned long(*FuncCreateMQServer)(const char*)`：创建消息队列服务器函数指针类型
  - `FuncDestroyMQServer _remover`：销毁消息队列服务器函数指针
    - `typedef void(*FuncDestroyMQServer)(unsigned long)`：销毁消息队列服务器函数指针类型
  - `FundPublishMessage _publisher`：发布消息函数指针
    - `typedef void(*FundPublishMessage)(unsigned long, const char*, const char*, unsigned long)`：发布消息函数指针类型
  - `FuncRegCallbacks _register`：注册回调函数指针
    - `typedef void(*FuncRegCallbacks)(FuncLogCallback)`：注册回调函数指针类型

- **异步处理**
  - `bool _stopped`：是否已停止标志
  - `boost::asio::io_service _asyncio`：Boost异步IO服务，用于异步事件处理
  - `StdThreadPtr _worker`：异步处理工作线程指针，用于后台处理事件队列

### 初始化 init

### 事件通知接口

#### 通知成交事件 notify

#### 通知订单事件 notify

#### 通知交易消息 notify

#### 通知日志事件 notify_log

#### 通知通用事件 notify_event

### 内部辅助方法

#### 将成交信息转换为JSON格式 tradeToJson

#### 将订单信息转换为JSON格式 orderToJson

## ShareManager.h/cpp — 共享内存管理器类
```cpp
class ShareManager
```

### 成员
- **初始化与状态**
  - `bool _inited`：是否已初始化标志
  - `bool _stopped`：是否已停止标志

- **共享内存域名称**
  - `std::string _exchg`：交换区名称
  - `std::string _sync`：同步区名称

- **参数监控**
  - `wt_hashmap<std::string, uint64_t> _secnames`：监控分区名称映射表
    - 键为分区名称（`std::string`），值为最后更新时间（`uint64_t`，微秒时间戳）
  - `StdThreadPtr _worker`：监控工作线程指针，用于后台监控参数变更
  - `WtUftEngine* _engine`：UFT引擎指针，用于参数变更通知

- **动态库管理**
  - `DllHandle _inst`：动态库句柄，用于管理WtShareHelper模块
  - `std::string _module`：模块路径，存储WtShareHelper模块的完整路径

- **初始化与域管理函数指针**
  - `func_init_master _init_master`：初始化主域函数指针
    - `typedef bool (*func_init_master)(const char*, const char*)`：初始化主域函数指针类型
  - `func_get_section_updatetime _get_section_updatetime`：获取分区更新时间函数指针
    - `typedef uint64_t(*func_get_section_updatetime)(const char*, const char*)`：获取分区更新时间函数指针类型
  - `func_commit_section _commit_section`：提交分区函数指针
    - `typedef bool(*func_commit_section)(const char*, const char*)`：提交分区函数指针类型

- **设置参数函数指针**
  - `func_set_string _set_string`：设置字符串类型参数函数指针
    - `typedef bool (*func_set_string)(const char*, const char*, const char*, const char*)`：设置字符串类型参数函数指针类型
  - `func_set_int32 _set_int32`：设置int32类型参数函数指针
    - `typedef bool (*func_set_int32)(const char*, const char*, const char*, int32_t)`：设置int32类型参数函数指针类型
  - `func_set_int64 _set_int64`：设置int64类型参数函数指针
    - `typedef bool (*func_set_int64)(const char*, const char*, const char*, int64_t)`：设置int64类型参数函数指针类型
  - `func_set_uint32 _set_uint32`：设置uint32类型参数函数指针
    - `typedef bool (*func_set_uint32)(const char*, const char*, const char*, uint32_t)`：设置uint32类型参数函数指针类型
  - `func_set_uint64 _set_uint64`：设置uint64类型参数函数指针
    - `typedef bool(*func_set_uint64)(const char*, const char*, const char*, uint64_t)`：设置uint64类型参数函数指针类型
  - `func_set_double _set_double`：设置double类型参数函数指针
    - `typedef bool(*func_set_double)(const char*, const char*, const char*, double)`：设置double类型参数函数指针类型

- **获取参数函数指针**
  - `func_get_string _get_string`：获取字符串类型参数函数指针
    - `typedef const char* (*func_get_string)(const char*, const char*, const char*, const char*)`：获取字符串类型参数函数指针类型
  - `func_get_int32 _get_int32`：获取int32类型参数函数指针
    - `typedef int32_t (*func_get_int32)(const char*, const char*, const char*, int32_t)`：获取int32类型参数函数指针类型
  - `func_get_int64 _get_int64`：获取int64类型参数函数指针
    - `typedef int64_t (*func_get_int64)(const char*, const char*, const char*, int64_t)`：获取int64类型参数函数指针类型
  - `func_get_uint32 _get_uint32`：获取uint32类型参数函数指针
    - `typedef uint32_t (*func_get_uint32)(const char*, const char*, const char*, uint32_t)`：获取uint32类型参数函数指针类型
  - `func_get_uint64 _get_uint64`：获取uint64类型参数函数指针
    - `typedef uint64_t (*func_get_uint64)(const char*, const char*, const char*, uint64_t)`：获取uint64类型参数函数指针类型
  - `func_get_double _get_double`：获取double类型参数函数指针
    - `typedef double (*func_get_double)(const char*, const char*, const char*, double)`：获取double类型参数函数指针类型

- **分配参数函数指针**
  - `func_allocate_string _allocate_string`：分配字符串类型参数函数指针
    - `typedef const char*(*func_allocate_string)(const char*, const char*, const char*, const char*, bool)`：分配字符串类型参数函数指针类型
  - `func_allocate_int32 _allocate_int32`：分配int32类型参数函数指针
    - `typedef int32_t* (*func_allocate_int32)(const char*, const char*, const char*, int32_t, bool)`：分配int32类型参数函数指针类型
  - `func_allocate_int64 _allocate_int64`：分配int64类型参数函数指针
    - `typedef int64_t* (*func_allocate_int64)(const char*, const char*, const char*, int64_t, bool)`：分配int64类型参数函数指针类型
  - `func_allocate_uint32 _allocate_uint32`：分配uint32类型参数函数指针
    - `typedef uint32_t* (*func_allocate_uint32)(const char*, const char*, const char*, uint32_t, bool)`：分配uint32类型参数函数指针类型
  - `func_allocate_uint64 _allocate_uint64`：分配uint64类型参数函数指针
    - `typedef uint64_t* (*func_allocate_uint64)(const char*, const char*, const char*, uint64_t, bool)`：分配uint64类型参数函数指针类型
  - `func_allocate_double _allocate_double`：分配double类型参数函数指针
    - `typedef double*	(*func_allocate_double)(const char*, const char*, const char*, double, bool)`：分配double类型参数函数指针类型

### 单例模式与核心属性

#### 获取单例实例 self

#### 设置UFT引擎指针 set_engine

### 初始化与生命周期管理

#### 初始化共享内存管理器 initialize

#### 启动参数监控 start_watching

#### 初始化共享内存域 init_domain

#### 提交参数监控分区 commit_param_watcher

### 参数设置接口

#### 设置字符串类型参数 set_value

#### 设置int32类型参数 set_value

#### 设置int64类型参数 set_value

#### 设置uint32类型参数 set_value

#### 设置uint64类型参数 set_value

#### 设置double类型参数 set_value

### 参数获取接口

#### 获取字符串类型参数 get_value

#### 获取int32类型参数 get_value

#### 获取int64类型参数 get_value

#### 获取uint32类型参数 get_value

#### 获取uint64类型参数 get_value

#### 获取double类型参数 get_value

### 参数分配接口（返回指针，支持直接修改）

#### 分配字符串类型字段 allocate_value

#### 分配int32类型字段 allocate_value

#### 分配int64类型字段 allocate_value

#### 分配uint32类型字段 allocate_value

#### 分配uint64类型字段 allocate_value

#### 分配double类型字段 allocate_value

## WtHelper.h/cpp — 辅助工具类
```cpp
class WtHelper
```

### 成员
- **时间状态（静态成员变量）**
  - `static uint32_t _cur_date`：当前日期（YYYYMMDD格式）
  - `static uint32_t _cur_time`：当前时间（HHMMSS格式），以分钟为准
  - `static uint32_t _cur_secs`：当前秒数（包含毫秒）
  - `static uint32_t _cur_tdate`：当前交易日（YYYYMMDD格式）

- **目录路径（静态成员变量）**
  - `static std::string _inst_dir`：实例所在目录
  - `static std::string _gen_dir`：生成文件输出目录

### 路径管理

#### 获取当前工作目录 getCWD

#### 获取模块路径 getModulePath

#### 获取基础目录 getBaseDir

#### 获取输出目录 getOutputDir

#### 获取策略数据目录 getStraDataDir

#### 获取策略用户数据目录 getStraUsrDatDir

#### 获取投资组合目录 getPortifolioDir

#### 获取实例目录 getInstDir

#### 设置实例目录 setInstDir

#### 设置生成目录 setGenerateDir

### 时间管理

#### 设置当前时间 setTime

#### 设置当前交易日 setTDate

#### 获取当前日期 getDate

#### 获取当前时间 getTime

#### 获取当前秒数 getSecs

#### 获取当前交易日 getTradingDate